# 08.04 · Advanced retrieval

**Chapter:** Retrieval-Augmented Generation · **Milestone:** M3

## Goals
- [ ] Query transformations: rewriting, multi-query, HyDE
- [ ] Contextual retrieval: enrich chunks with document context before indexing
- [ ] Small-to-big (parent-document) retrieval
- [ ] Measure every trick against the baseline — most don't help everywhere

## Resources
- [Anthropic – Contextual Retrieval](https://www.anthropic.com/news/contextual-retrieval) — free
- [HyDE paper (Precise Zero-Shot Dense Retrieval)](https://arxiv.org/abs/2212.10496) — free
- [LlamaIndex – Advanced retrieval strategies](https://docs.llamaindex.ai/en/stable/optimizing/advanced_retrieval/advanced_retrieval/) — free

In [ ]:
import time

import numpy as np

from aieng import rag
from aieng.llm import chat  # every call below sets max_tokens: small models sometimes ramble until timeout

docs = rag.corpus()
evalset = rag.eval_set()[:60]  # each technique adds LLM calls: keep the eval small and fast
print(len(evalset), "eval questions")


def dense_ids(query, k):
    return [pid for pid, _, _ in rag.search(query, k)]


def rrf(rankings, k=60):
    s = {}
    for ranking in rankings:
        for rank, d in enumerate(ranking):
            s[d] = s.get(d, 0) + 1 / (k + rank + 1)
    return sorted(s, key=s.get, reverse=True)


def recall(search_fn, k=5):
    start = time.perf_counter()
    r = np.mean([bool(set(search_fn(row["question"])[:k]) & row["gold"]) for row in evalset])
    return r, (time.perf_counter() - start) / len(evalset)


results = {"baseline dense": recall(lambda q: dense_ids(q, 5))}
print(results)

## 1. Query rewriting
Users type vague, chatty or typo-ridden queries. An LLM rewrites them into a clean search query. Most useful in **conversations**
("what about his wife?" → "Abraham Lincoln wife") — that's where it shines; on clean questions it can hurt.

In [ ]:
def rewrite(q):
    return chat(f"Rewrite this as a short, keyword-rich search query. Reply with the query only.\n\n{q}", temperature=0, max_tokens=60).strip()


print(rewrite("hey so like when did the lincoln guy get shot??"))
results["rewrite"] = recall(lambda q: dense_ids(rewrite(q), 5))

## 2. Multi-query
Generate several phrasings, retrieve for each, fuse with RRF (07.04). Covers vocabulary mismatch at the cost of N searches + 1 LLM call.

In [ ]:
def multi_query(q, n=3):
    out = chat(f"Write {n} different search queries that would find the answer to this question, one per line, no numbering.\n\n{q}", temperature=0.5, max_tokens=150)
    variants = [v.strip("-• ").strip() for v in out.splitlines() if v.strip()][:n]
    return rrf([dense_ids(v, 20) for v in [q, *variants]])


results["multi-query + RRF"] = recall(multi_query)

## 3. HyDE: search with a hypothetical answer
Questions and answers look different; passages look like answers. Let the LLM write a plausible (possibly wrong!) answer passage and embed
*that*. Works well when the model knows the domain's style; risky when it confidently writes the wrong facts.

In [ ]:
def hyde(q):
    fake = chat(f"Write a short encyclopedia-style paragraph answering: {q}", temperature=0, max_tokens=200)
    v = rag.embed([fake])[0]  # embed as a document, not a query
    return list(np.argsort(-(rag.index() @ v))[:20])


def hyde_plus(q):  # fuse HyDE with the normal query: safer
    return rrf([dense_ids(q, 20), hyde(q)])


results["HyDE"] = recall(hyde)
results["HyDE + query (RRF)"] = recall(hyde_plus)

## 4. Small-to-big: match small, return big
Embed **sentences** (precise matching), but return the **whole passage** (enough context to answer). Here: map each sentence back to its
passage id. Same idea as LlamaIndex's "sentence window" / LangChain's ParentDocumentRetriever.

In [ ]:
import re

sentences, parent = [], []
for pid, d in enumerate(docs):
    for s in re.split(r"(?<=[.!?])\s+", d):
        if len(s) > 30:
            sentences.append(s); parent.append(pid)
S = rag.embed(sentences)
parent = np.array(parent)
print(f"{len(sentences)} sentences from {len(docs)} passages")


def small_to_big(q, k=5):
    order = np.argsort(-(S @ rag.embed([q], query=True)[0]))
    seen = []
    for p in parent[order]:
        if p not in seen:
            seen.append(int(p))
        if len(seen) == k:
            break
    return seen


results["small-to-big (sentences)"] = recall(small_to_big)

## 5. Contextual retrieval
Chunks lose context ("He was elected in 1860" — who?). Before embedding, prepend a 1–2 sentence LLM-written context situating the chunk in
its document. Anthropic reports large failure-rate reductions combined with BM25 + reranking. Cost: one LLM call per chunk at index time
(cheap with prompt caching, since the document prefix repeats).

Demo on a slice (each passage contextualised with its neighbours, since mini-wiki has no document boundaries):

In [ ]:
def contextualize(pid, window=3):
    neighbours = " ".join(docs[max(0, pid - window) : pid + window + 1])
    return chat(f"<document>\n{neighbours}\n</document>\nHere is a chunk from the document:\n<chunk>\n{docs[pid]}\n</chunk>\n"
                "Write one short sentence that situates this chunk within the document (who/what it is about). Reply with that sentence only.",
                temperature=0, max_tokens=80).strip()


for pid in [290, 291]:
    print(f"[{pid}] original: {docs[pid][:90]}...\n      context : {contextualize(pid)}\n")

Full evaluation would contextualise all 3,200 passages (~3,200 LLM calls: minutes with a fast hosted model + caching, much longer on a laptop).
That's exercise 2.

## 6. Scoreboard

In [ ]:
for name, (r, secs) in results.items():
    print(f"{name:26} recall@5 {r:.3f} | {secs:5.2f} s/query")

Typical outcome: on clean, well-formed questions like these, transformations give small or *negative* gains and add latency; they earn
their keep on messy real-user queries and conversational follow-ups. Hybrid + rerank (07.05) is usually the better first investment.
**Never ship a trick because a blog post said so — ship it because your eval says so.**

## Exercises
1. Make the eval harder: rewrite 20 eval questions to be vague/conversational ("and when did he die?" after a Lincoln question). Re-run all methods.
2. Contextualise all passages with a hosted model (Gemini Flash-Lite/Groq) using async + a semaphore (05.05), re-embed, and measure recall.
3. Combine the best two techniques with the reranker from 07.05. What's the best recall@5 you can reach, and at what latency?

In [ ]:
# your code here